# Kafka

Kafka is a distributed event-streaming platform used to publish, store, and consume streams of events in real time. Think of it as a highly scalable, durable message bus.

You must be able to explain these five terms clearly:

**Producer:** The application that sends (publishes) messages to Kafka.

**Consumer:** The application that reads (subscribes to) messages from Kafka.

**Topic:** A logical category (like a folder) where messages are stored. Example: `orders_topic`.

**Partition:** A topic is split into partitions for parallelism and scalability. Each partition is an ordered, immutable log.

**Offset:** A unique sequential ID for each message within a partition. Consumers track offsets to know what they have already processed.

"Kafka is a distributed pub-sub system. Producers write messages to Topics, which are split into Partitions for parallelism. Consumers read from partitions and track their progress using Offsets."

## PySpark – Read Stream from Kafka

```python
%python
from pyspark.sql.functions import col

# 1. Read the stream from Kafka
raw_kafka_df = (spark.readStream
  .format("kafka")
  .option("kafka.bootstrap.servers", "your-kafka-broker:9092")
  .option("subscribe", "sportsbook_bets_topic")    #Spark, listen to the topic called bets
  .option("startingOffsets", "latest")   #read only new messages
  .load()
)

# 2. Kafka sends data as binary. Cast value to String
bronze_df = raw_kafka_df.select(
    col("value").cast("string").alias("raw_json"),
    col("timestamp").alias("kafka_timestamp")
)

# 3. Write to Bronze Delta Table
(bronze_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/checkpoints/bronze/bets")
  .trigger(availableNow=True) # Or continuous if needed
  .toTable("workspace.default.bronze_sportsbook_bets")
)


silver_df = (parsed
    .withWatermark("event_time", "10 minutes")
    .dropDuplicates(["bet_id", "event_time"])
    .withColumn("silver_load_time", current_timestamp())
```

## Watermark

A watermark is a time threshold for how long the system will wait for late-arriving data before finalizing a result and dropping old state.

**Step 1: `.withWatermark("event_time", "10 minutes")`**

You are telling Spark: "Look at the `event_time` column. If a record is more than 10 minutes late, ignore it."

**Step 2: `.dropDuplicates(["bet_id", "event_time"])`**

You are telling Spark: "Remove duplicate bets, but only look back 10 minutes."

**Batch = process data in big chunks on a schedule. Streaming = process data continuously as it arrives.**